[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_03_Tool_Use_Function_Calling.ipynb)

# 🛠️ Lesson 3: Tool Use & Function Calling
### *Giving LLMs the Power to Act in the World*

---

**Curriculum:** AI Engineering from Scratch  
**Prerequisites:** Lesson 1 (LLM Fundamentals) · Lesson 2 (Prompt Engineering)  
**Time:** ~60 minutes

---

## 🎯 What You'll Learn

By the end of this lesson you'll understand:

1. **Why tool use exists** — the fundamental limitation LLMs have without it
2. **How tool use works mechanically** — the request/response cycle
3. **How to define tools** for Claude using JSON Schema
4. **How to execute the tool loop** — the core pattern every AI agent is built on
5. **Real examples** — calculator, weather lookup, and a multi-tool agent

---

## 🧠 Concept: Why Do LLMs Need Tools?

LLMs are **frozen in time**. When you trained Claude or GPT, their knowledge was cut off at a certain date. More importantly, they exist purely as *text predictors* — they cannot:

- Look up real-time data (stock prices, weather, news)
- Do precise arithmetic reliably (they hallucinate math!)
- Write to a database or send an email
- Execute code and observe the result
- Search the web

**Tool use (also called "function calling") bridges this gap.**

Instead of the LLM trying to answer everything from memory, it can *say*: "I need to call a function to get this information. Here are the arguments I want to pass." Your code then executes the function and returns the result to the LLM, which uses it to form a final answer.

This is the **core primitive of AI agents**. An agent is essentially an LLM in a loop, calling tools until the task is done.

---

## 🔄 The Tool Use Flow (Commit This to Memory)

```
YOU                          CLAUDE (API)              YOUR TOOL CODE
────────────────────────────────────────────────────────────────────
1. Send message + tool defs  ──────────────────────►
                                                      
                             ◄──────────────────────  2. Claude says: "call
                               stop_reason='tool_use'    get_weather(city='NYC')"

3. YOU run get_weather('NYC')                         ◄── your Python function
   → returns "72°F, sunny"

4. Send result back          ──────────────────────►
   as tool_result message

                             ◄──────────────────────  5. Claude gives final
                               stop_reason='end_turn'    human answer: "NYC is
                                                         72°F and sunny today!"
```

**Key insight:** Claude never actually *runs* the code. It generates structured JSON describing *what* to call and *with what arguments*. YOU run the code. This is intentional — it keeps the LLM sandboxed.

---

In [ ]:
# ============================================================
# SETUP — Run this cell first every time you open in Colab
# ============================================================
!pip install anthropic -q

import anthropic
import json

# Load your API key from Colab Secrets
# How to set it up (one-time):
#   1. Click the 🔑 key icon in the left sidebar
#   2. Add a secret named: ANTHROPIC_API_KEY
#   3. Paste your key from https://console.anthropic.com/
#   4. Toggle "Notebook access" ON
from google.colab import userdata
ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
print("✅ Setup complete! Anthropic client ready.")

---

## Part 1: Defining a Tool

A tool definition has three parts:

| Field | What it is |
|-------|------------|
| `name` | The function name (snake_case, no spaces) |
| `description` | Plain English: what does this tool do, when should Claude use it? |
| `input_schema` | JSON Schema object describing the parameters Claude must provide |

The `description` is the most important part. Claude reads it to decide *whether* to call the tool. Write it like you're writing documentation for a smart colleague.

Let's define our first tool — a simple calculator:

In [ ]:
# ============================================================
# PART 1: Defining a Tool
# ============================================================

# Step 1: Define the tool spec (what Claude sees)
calculator_tool = {
    "name": "calculate",
    "description": (
        "Performs precise arithmetic calculations. "
        "Use this whenever the user asks for a calculation involving numbers. "
        "Supports addition, subtraction, multiplication, division, and exponentiation."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "expression": {
                "type": "string",
                "description": "A valid Python math expression, e.g. '2 ** 32' or '(100 * 1.08) / 12'"
            }
        },
        "required": ["expression"]
    }
}

# Step 2: Define the actual Python function that EXECUTES the tool
def calculate(expression: str) -> str:
    """The real function your code runs when Claude calls this tool."""
    try:
        # eval() is safe here because Claude generates the expression
        # In production you'd use a math parsing library (e.g., simpleeval)
        result = eval(expression, {"__builtins__": {}}, {})
        return str(result)
    except Exception as e:
        return f"Error: {e}"

# Quick sanity check
print("Tool spec:", json.dumps(calculator_tool, indent=2))
print("\nFunction test:", calculate("2 ** 32"))

---

## Part 2: The Tool Loop (Step by Step)

Now let's execute the full loop manually so you can see every step. We'll ask Claude a math question, watch it request the tool, run the tool, and feed the result back.

In [ ]:
# ============================================================
# PART 2: The Tool Loop — Manual Step-by-Step
# ============================================================

user_question = "What is 2 to the power of 32? And then divide that result by 1000?"

# ── STEP 1: Initial request to Claude ─────────────────────────────────────
print("=" * 60)
print("STEP 1: Sending question to Claude with tool definition")
print("=" * 60)

response = client.messages.create(
    model="claude-opus-4-5",
    max_tokens=1024,
    tools=[calculator_tool],        # <-- Claude knows about this tool
    messages=[
        {"role": "user", "content": user_question}
    ]
)

print(f"stop_reason: {response.stop_reason}")
print(f"content blocks: {response.content}")

# ── STEP 2: Check if Claude wants to use a tool ───────────────────────────
print("\n" + "=" * 60)
print("STEP 2: Did Claude call a tool?")
print("=" * 60)

if response.stop_reason == "tool_use":
    print("✅ Yes! Claude wants to use a tool.")
    
    # Find the tool_use block in the response
    tool_use_block = next(
        block for block in response.content 
        if block.type == "tool_use"
    )
    
    tool_name = tool_use_block.name
    tool_input = tool_use_block.input
    tool_use_id = tool_use_block.id  # needed to match result back
    
    print(f"  Tool name: {tool_name}")
    print(f"  Arguments: {json.dumps(tool_input, indent=4)}")
    print(f"  Tool use ID: {tool_use_id}")
else:
    print("Claude answered directly without tools.")

In [ ]:
# ── STEP 3: YOUR CODE executes the tool ───────────────────────────────────
print("=" * 60)
print("STEP 3: WE run the actual function")
print("=" * 60)

# Dispatch to the right function based on tool name
tool_registry = {
    "calculate": calculate
}

tool_result = tool_registry[tool_name](**tool_input)
print(f"  Result from our function: {tool_result}")

# ── STEP 4: Send result back to Claude ────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 4: Sending result back to Claude")
print("=" * 60)

# We must maintain the full conversation history
# Claude's response becomes the assistant turn,
# and we add our tool result as the next user turn
final_response = client.messages.create(
    model="claude-opus-4-5",
    max_tokens=1024,
    tools=[calculator_tool],
    messages=[
        # Original user question
        {"role": "user", "content": user_question},
        # Claude's tool-calling response (must include full content block)
        {"role": "assistant", "content": response.content},
        # Our tool result
        {
            "role": "user",
            "content": [
                {
                    "type": "tool_result",
                    "tool_use_id": tool_use_id,   # must match!
                    "content": tool_result
                }
            ]
        }
    ]
)

print(f"stop_reason: {final_response.stop_reason}")
print("\n" + "=" * 60)
print("STEP 5: Claude's final answer to the user")
print("=" * 60)
print(final_response.content[0].text)

---

## Part 3: A Reusable Tool Loop Helper

Writing the full loop every time is tedious. Let's build a helper function that handles the entire tool execution cycle automatically — this is the pattern you'll use in real agents.

In [ ]:
# ============================================================
# PART 3: A Reusable Tool Loop Helper
# ============================================================

def run_agent(user_message: str, tools: list, tool_registry: dict, 
              model: str = "claude-opus-4-5", verbose: bool = True) -> str:
    """
    Runs an agent loop: keeps calling Claude and executing tools
    until Claude gives a final answer (stop_reason == 'end_turn').
    
    Args:
        user_message: The user's input
        tools: List of tool definition dicts (what Claude sees)
        tool_registry: Dict mapping tool name → Python callable
        model: Claude model to use
        verbose: Print intermediate steps if True
    
    Returns:
        Claude's final text response
    """
    messages = [{"role": "user", "content": user_message}]
    
    if verbose:
        print(f"👤 User: {user_message}\n")
    
    # Agent loop — keeps going until stop_reason is 'end_turn'
    iteration = 0
    while True:
        iteration += 1
        
        response = client.messages.create(
            model=model,
            max_tokens=1024,
            tools=tools,
            messages=messages
        )
        
        # Add Claude's response to conversation history
        messages.append({"role": "assistant", "content": response.content})
        
        if response.stop_reason == "end_turn":
            # Claude is done — extract the text and return
            final_text = next(
                (block.text for block in response.content if hasattr(block, "text")),
                "[No text response]"
            )
            if verbose:
                print(f"🤖 Claude: {final_text}")
            return final_text
        
        elif response.stop_reason == "tool_use":
            # Execute ALL tool calls in this response (Claude can request multiple)
            tool_results = []
            
            for block in response.content:
                if block.type != "tool_use":
                    continue
                
                tool_name = block.name
                tool_input = block.input
                tool_use_id = block.id
                
                if verbose:
                    print(f"🔧 [Iteration {iteration}] Claude calls: {tool_name}({tool_input})")
                
                # Execute the tool
                if tool_name in tool_registry:
                    result = tool_registry[tool_name](**tool_input)
                else:
                    result = f"Error: unknown tool '{tool_name}'"
                
                if verbose:
                    print(f"   ↳ Result: {result}")
                
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": tool_use_id,
                    "content": str(result)
                })
            
            # Add all tool results as the next user message
            messages.append({"role": "user", "content": tool_results})
        
        else:
            # Unexpected stop reason
            break
    
    return "[Agent loop ended unexpectedly]"

print("✅ run_agent() helper defined! Let's use it.")

In [ ]:
# Test it with the calculator
result = run_agent(
    user_message="If I invest $5,000 at 8% annual return compounded yearly for 20 years, how much will I have?",
    tools=[calculator_tool],
    tool_registry={"calculate": calculate}
)

# 💡 EXPERIMENT: Change the investment amount, interest rate, or years.
# Watch how Claude generates the correct expression for the compound interest formula.

---

## Part 4: Multiple Tools

Real agents have many tools. Claude will choose the right one based on the descriptions. Let's build a tiny agent with a weather tool and a calculator, and ask it a question that needs both.

In [ ]:
# ============================================================
# PART 4: Multiple Tools — Claude Chooses the Right One
# ============================================================

# Tool 2: Simulated weather lookup
# (In real life this would call a weather API like OpenWeatherMap)
weather_tool = {
    "name": "get_weather",
    "description": (
        "Returns the current weather for a given city. "
        "Use this when the user asks about weather conditions in any city."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "City name, e.g. 'New York' or 'Tokyo'"
            },
            "unit": {
                "type": "string",
                "enum": ["celsius", "fahrenheit"],
                "description": "Temperature unit. Defaults to celsius."
            }
        },
        "required": ["city"]
    }
}

def get_weather(city: str, unit: str = "celsius") -> str:
    """Simulated weather function. In production, call a real weather API."""
    # Fake data — replace with: requests.get(f"https://api.openweathermap.org/...")
    fake_data = {
        "new york":  {"celsius": 22, "fahrenheit": 72, "condition": "Partly cloudy"},
        "london":    {"celsius": 14, "fahrenheit": 57, "condition": "Rainy"},
        "tokyo":     {"celsius": 28, "fahrenheit": 82, "condition": "Sunny"},
        "mumbai":    {"celsius": 35, "fahrenheit": 95, "condition": "Hot and humid"},
        "sydney":    {"celsius": 19, "fahrenheit": 66, "condition": "Clear"},
    }
    city_key = city.lower()
    if city_key in fake_data:
        data = fake_data[city_key]
        temp = data[unit]
        symbol = "°C" if unit == "celsius" else "°F"
        return f"{city}: {temp}{symbol}, {data['condition']}"
    return f"Weather data not available for {city}"

# Tool 3: Unit converter
unit_converter_tool = {
    "name": "convert_units",
    "description": (
        "Converts between common units of measurement. "
        "Supports: temperature (celsius/fahrenheit/kelvin), "
        "distance (km/miles/meters/feet), weight (kg/lbs/grams)."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "value": {"type": "number", "description": "The value to convert"},
            "from_unit": {"type": "string", "description": "Source unit, e.g. 'celsius'"},
            "to_unit": {"type": "string", "description": "Target unit, e.g. 'fahrenheit'"}
        },
        "required": ["value", "from_unit", "to_unit"]
    }
}

def convert_units(value: float, from_unit: str, to_unit: str) -> str:
    conversions = {
        ("celsius", "fahrenheit"): lambda v: v * 9/5 + 32,
        ("fahrenheit", "celsius"): lambda v: (v - 32) * 5/9,
        ("celsius", "kelvin"):     lambda v: v + 273.15,
        ("kelvin", "celsius"):     lambda v: v - 273.15,
        ("km", "miles"):           lambda v: v * 0.621371,
        ("miles", "km"):           lambda v: v * 1.60934,
        ("kg", "lbs"):             lambda v: v * 2.20462,
        ("lbs", "kg"):             lambda v: v / 2.20462,
        ("meters", "feet"):        lambda v: v * 3.28084,
        ("feet", "meters"):        lambda v: v / 3.28084,
    }
    key = (from_unit.lower(), to_unit.lower())
    if key in conversions:
        result = conversions[key](value)
        return f"{value} {from_unit} = {result:.2f} {to_unit}"
    return f"Don't know how to convert {from_unit} to {to_unit}"

print("✅ Three tools defined: calculator, weather, unit converter")
print("\nWeather test:", get_weather("Tokyo", "celsius"))
print("Converter test:", convert_units(100, "km", "miles"))

In [ ]:
# Now ask a question that requires MULTIPLE tool calls
all_tools = [calculator_tool, weather_tool, unit_converter_tool]
all_registry = {
    "calculate": calculate,
    "get_weather": get_weather,
    "convert_units": convert_units
}

print("─" * 60)
run_agent(
    user_message="What's the weather in Tokyo right now? Convert it to Fahrenheit too. Also, what is 37 * 48?",
    tools=all_tools,
    tool_registry=all_registry
)

# 💡 EXPERIMENT: Try asking about the weather in Mumbai, London, or Sydney.
# Try asking multiple math questions at once and watch Claude call the tool multiple times.

---

## Part 5: Tool Error Handling

What happens when a tool fails? Claude is smart enough to handle errors gracefully if you tell it what went wrong. Let's see how to communicate errors back.

In [ ]:
# ============================================================
# PART 5: Tool Error Handling
# ============================================================

# You can pass errors back via tool_result too
# Claude will read the error and either retry with different args or explain the issue

def run_agent_with_error_handling(user_message: str, tools: list, 
                                   tool_registry: dict) -> str:
    """Extended run_agent that explicitly marks errors in tool results."""
    messages = [{"role": "user", "content": user_message}]
    print(f"👤 User: {user_message}\n")
    
    while True:
        response = client.messages.create(
            model="claude-opus-4-5",
            max_tokens=1024,
            tools=tools,
            messages=messages
        )
        
        messages.append({"role": "assistant", "content": response.content})
        
        if response.stop_reason == "end_turn":
            final = next(
                (b.text for b in response.content if hasattr(b, "text")), ""
            )
            print(f"🤖 Claude: {final}")
            return final
        
        if response.stop_reason == "tool_use":
            tool_results = []
            for block in response.content:
                if block.type != "tool_use":
                    continue
                
                print(f"🔧 Claude calls: {block.name}({block.input})")
                
                try:
                    result = tool_registry[block.name](**block.input)
                    is_error = False
                    print(f"   ↳ Success: {result}")
                except Exception as e:
                    result = f"Tool execution failed: {str(e)}"
                    is_error = True
                    print(f"   ↳ Error: {result}")
                
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": str(result),
                    "is_error": is_error  # Tell Claude this was an error
                })
            
            messages.append({"role": "user", "content": tool_results})
            continue
        break
    return ""

# Try asking about a city we don't have data for — see how Claude handles the error
print("─" * 60)
run_agent_with_error_handling(
    user_message="What's the weather like in Bangalore right now?",
    tools=[weather_tool],
    tool_registry={"get_weather": get_weather}
)

# 💡 EXPERIMENT: Try 'Paris', 'Berlin', or any city not in our fake_data dict.
# Notice how Claude acknowledges the limitation gracefully.

---

## Part 6: Practical Exercise — Build a Mini Research Agent

Now it's your turn. You'll build a mini research assistant that can:
- Look up a topic (simulated search)
- Summarize findings
- Do related calculations if needed

This is the closest thing to a real agent you've built yet.

In [ ]:
# ============================================================
# PART 6: Exercise — Mini Research Agent
# ============================================================

# A simulated knowledge base (in real life: call a search API or vector DB)
KNOWLEDGE_BASE = {
    "transformer": """The Transformer is a neural network architecture introduced in 2017 by Vaswani et al. 
        in 'Attention Is All You Need'. It relies entirely on self-attention mechanisms, 
        dispensing with recurrence and convolutions. It became the foundation of models like BERT, GPT, and Claude.
        Training data size (GPT-3): 570 GB. Parameters: 175 billion.""",
    
    "rag": """Retrieval-Augmented Generation (RAG) is a technique that combines a language model 
        with a retrieval system. Instead of relying solely on training data, the model retrieves 
        relevant documents at inference time and uses them as context. 
        Introduced by Lewis et al. in 2020. Reduces hallucinations significantly.""",
    
    "reinforcement learning from human feedback": """RLHF is a fine-tuning technique that uses human 
        preference data to align model outputs with human values. A reward model is trained on 
        human comparisons, then PPO (Proximal Policy Optimization) is used to optimize the LLM. 
        Used in ChatGPT, Claude, and most production LLMs.""",
    
    "vector database": """A vector database stores embeddings (dense numerical representations of text/images). 
        Similarity search uses cosine or dot-product distance. Popular options: Pinecone, Weaviate, Chroma, pgvector. 
        Used in RAG systems, semantic search, and recommendation engines."""
}

# Define the search tool
search_tool = {
    "name": "search_knowledge_base",
    "description": (
        "Search for information about AI/ML topics. "
        "Use this to look up technical concepts before answering. "
        "Available topics: transformer, rag, reinforcement learning from human feedback, vector database."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The topic or concept to look up"
            }
        },
        "required": ["query"]
    }
}

def search_knowledge_base(query: str) -> str:
    query_lower = query.lower()
    for key, value in KNOWLEDGE_BASE.items():
        if key in query_lower or query_lower in key:
            return value
    return f"No information found for '{query}'. Available: {list(KNOWLEDGE_BASE.keys())}"

# Build the research agent
research_tools = [search_tool, calculator_tool]
research_registry = {
    "search_knowledge_base": search_knowledge_base,
    "calculate": calculate
}

print("🔬 Mini Research Agent Ready")
print("─" * 60)

# Ask a question that requires searching + reasoning
run_agent(
    user_message="""Explain what a Transformer is, and tell me: 
    if GPT-3 has 175 billion parameters and each parameter takes 4 bytes (float32), 
    how many gigabytes is that?""",
    tools=research_tools,
    tool_registry=research_registry
)

# 💡 EXPERIMENT IDEAS:
# 1. Ask: "What is RAG and how does it relate to vector databases?"
# 2. Ask: "Explain RLHF in simple terms"
# 3. Try to add a new topic to KNOWLEDGE_BASE and ask about it
# 4. Add a tool that counts the number of words in the retrieved text

---

## 🎓 Summary

Here's what you learned today:

**Why tool use exists**  
LLMs are text predictors, frozen in time. Tools give them access to real-time data, computation, and external systems.

**How the tool loop works**  
1. You send a message + tool definitions  
2. Claude responds with `stop_reason='tool_use'` and a structured JSON call  
3. YOUR code executes the function  
4. You send the result back as a `tool_result` message  
5. Claude gives a final answer (`stop_reason='end_turn'`)

**Key patterns you built**  
- Tool definition with JSON Schema  
- Manual step-by-step loop  
- Reusable `run_agent()` helper with multi-tool support  
- Error handling via `is_error` flag  
- A working mini research agent

---

## 🚀 What's Next

**Lesson 4: Building Your First Real Agent**

You've seen the tool loop. Next, we add the **planning layer** — the ReAct (Reasoning + Acting) pattern where the agent thinks about *what* to do before acting, and keeps iterating until a goal is fully achieved. We'll build an agent that can autonomously break down a complex task, use multiple tools in sequence, and self-correct when things go wrong.

---

## 🔗 Useful Links

- [Anthropic Tool Use Docs](https://docs.anthropic.com/en/docs/build-with-claude/tool-use) — official reference  
- [JSON Schema Guide](https://json-schema.org/learn/getting-started-step-by-step) — for writing tool input schemas  
- [OpenWeatherMap API](https://openweathermap.org/api) — to replace the fake weather function (free tier available)  
